# Deploy Custom Deep Research System on OpenShift

Deploy the complete document research system to OpenShift using plain Kubernetes
manifests or Helm chart — no custom operators required.

## 1. Load Environment

In [ ]:
import os
import subprocess
import json
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

NAMESPACE = os.getenv("NAMESPACE", "doc-research-lab")
REGISTRY = os.getenv("REGISTRY", "quay.io/your-org")
IMAGE_TAG = os.getenv("IMAGE_TAG", "latest")

print(f"Namespace:  {NAMESPACE}")
print(f"Registry:   {REGISTRY}")
print(f"Image tag:  {IMAGE_TAG}")
print("\u2705 Environment loaded")

## 2. Verify Cluster Access

Confirm we can reach the OpenShift cluster and have permissions to deploy.

In [ ]:
r = subprocess.run(["oc", "whoami"], capture_output=True, text=True)
if r.returncode == 0:
    print(f"Logged in as: {r.stdout.strip()}")
else:
    print("\u274c Not logged in. Run: oc login <cluster-url>")
    raise SystemExit(1)

r = subprocess.run(
    ["oc", "auth", "can-i", "create", "deployments", "-n", NAMESPACE],
    capture_output=True, text=True,
)
if "yes" in r.stdout.lower():
    print(f"\u2705 Permission to deploy in namespace '{NAMESPACE}'")
else:
    print(f"\u26a0\ufe0f May lack permissions in '{NAMESPACE}' — namespace will be created if needed")

## 3. Build Container Images

Build all 6 container images (4 MCP servers + backend + frontend) using Podman.

In [ ]:
project_root = os.path.dirname(os.getcwd())

r = subprocess.run(
    ["make", "build-all"],
    cwd=project_root,
    capture_output=True, text=True,
)

print(r.stdout)
if r.returncode == 0:
    print("\u2705 All container images built successfully")
else:
    print(r.stderr)
    print("\u274c Build failed — check Dockerfiles and dependencies")

## 4. Push Images to Registry

Tag and push all images to the container registry defined in `REGISTRY`.

In [ ]:
if REGISTRY == "quay.io/your-org":
    print("\u26a0\ufe0f REGISTRY is still the default placeholder.")
    print("   Set REGISTRY in .env to your actual registry before pushing.")
else:
    r = subprocess.run(
        ["make", "push-all"],
        cwd=project_root,
        env={**os.environ, "REGISTRY": REGISTRY, "IMAGE_TAG": IMAGE_TAG},
        capture_output=True, text=True,
    )
    print(r.stdout)
    if r.returncode == 0:
        print(f"\u2705 All images pushed to {REGISTRY}")
    else:
        print(r.stderr)
        print("\u274c Push failed — verify registry credentials (podman login)")

## 5. Deploy with Helm (Recommended)

Deploy the full application stack as a single Helm release.

In [ ]:
helm_chart = os.path.join(project_root, "deploy", "helm", "doc-research")

if not os.path.isdir(helm_chart):
    print(f"\u26a0\ufe0f Helm chart not found at {helm_chart}")
    print("   Use Method B (plain manifests) below, or create the Helm chart first.")
else:
    cmd = [
        "helm", "upgrade", "--install", "doc-research",
        helm_chart,
        "-n", NAMESPACE, "--create-namespace",
        "--set", f"global.image.registry={REGISTRY}",
        "--set", f"global.image.tag={IMAGE_TAG}",
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode == 0:
        print(f"\u2705 Helm release 'doc-research' deployed to namespace '{NAMESPACE}'")
    else:
        print(r.stderr)
        print("\u274c Helm deployment failed")

## 6. Alternative: Deploy with Plain Manifests

If not using Helm, deploy infrastructure and apps separately via `make` targets.

In [ ]:
# Uncomment the lines below to deploy via plain manifests instead of Helm.
# This deploys: namespace, secret, PostgreSQL, MinIO, then all application pods.

# r = subprocess.run(
#     ["make", "deploy-infra"],
#     cwd=project_root,
#     capture_output=True, text=True,
# )
# print(r.stdout)
# if r.returncode != 0:
#     print(r.stderr)
#     raise RuntimeError("Infrastructure deployment failed")
#
# r = subprocess.run(
#     ["make", "deploy-apps"],
#     cwd=project_root,
#     capture_output=True, text=True,
# )
# print(r.stdout)
# if r.returncode == 0:
#     print("\u2705 All manifests applied")
# else:
#     print(r.stderr)
#     print("\u274c Manifest deployment failed")

## 7. Verify Pods Are Running

Wait for all pods to reach Running state.

In [ ]:
import time

expected_apps = [
    "backend", "frontend",
    "vector-search-mcp", "web-search-mcp",
    "verification-mcp", "observability-mcp",
    "postgresql", "minio",
]

print(f"Waiting for pods in namespace '{NAMESPACE}'...\n")
time.sleep(5)

r = subprocess.run(
    ["oc", "get", "pods", "-n", NAMESPACE,
     "-o", "jsonpath={range .items[*]}{.metadata.name}{'\\t'}{.status.phase}{'\\n'}{end}"],
    capture_output=True, text=True,
)

if r.returncode == 0 and r.stdout.strip():
    all_running = True
    for line in r.stdout.strip().split("\n"):
        if line:
            parts = line.split("\t")
            name = parts[0]
            phase = parts[1] if len(parts) > 1 else "Unknown"
            icon = "\u2705" if phase == "Running" else "\u23f3"
            print(f"  {icon} {name}: {phase}")
            if phase != "Running":
                all_running = False
    print()
    if all_running:
        print("\u2705 All pods are running")
    else:
        print("\u23f3 Some pods still starting — re-run this cell in a moment")
else:
    print(f"\u26a0\ufe0f No pods found in '{NAMESPACE}'. Deployment may still be rolling out.")
    if r.stderr:
        print(r.stderr)

## 8. Get Application Routes

Retrieve the externally accessible URLs for backend and frontend.

In [ ]:
r = subprocess.run(
    ["oc", "get", "routes", "-n", NAMESPACE,
     "-o", "jsonpath={range .items[*]}{.metadata.name}{'\\t'}{.spec.host}{'\\n'}{end}"],
    capture_output=True, text=True,
)

if r.returncode == 0 and r.stdout.strip():
    print("Application routes:\n")
    for line in r.stdout.strip().split("\n"):
        if line:
            name, host = line.split("\t")
            print(f"  {name}: https://{host}")
    print("\n\u2705 Routes retrieved")
else:
    print("\u26a0\ufe0f No routes found. Create them if external access is needed:")
    print(f"   oc expose svc/backend -n {NAMESPACE}")
    print(f"   oc expose svc/frontend -n {NAMESPACE}")